In [1]:
with open('input.txt', 'r') as file:
    data = [line.strip().split('~') for line in file.readlines()]
    data = [(tuple(map(int, start_pos.split(','))), tuple(map(int, end_pos.split(',')))) for start_pos, end_pos in data]

In [2]:
def solve_puzzles():
    # Parse and sort bricks by lowest Z coordinate
    bricks = []
    for start_coordinate, end_coordinate in data:
        start_x, start_y, start_z = start_coordinate
        end_x, end_y, end_z = end_coordinate
        bricks.append((
            min(start_x, end_x), min(start_y, end_y), min(start_z, end_z),
            max(start_x, end_x), max(start_y, end_y), max(start_z, end_z)
        ))
    
    bricks.sort(key=lambda brick: brick[2])

    # Simulate falling bricks
    height_map = {} # (x, y) -> (max_height, brick_index)
    settled_bricks = []
    supported_by = {} # brick_index -> set of supporter_indices
    supports = {}     # brick_index -> set of supported_indices

    for brick_index, (start_x, start_y, start_z, end_x, end_y, end_z) in enumerate(bricks):
        # Find the highest point below this brick
        current_max_height = 0

        for x in range(start_x, end_x + 1):
            for y in range(start_y, end_y + 1):
                height, _ = height_map.get((x, y), (0, -1))

                if height > current_max_height:
                    current_max_height = height
        
        # Calculate new Z position
        brick_height = end_z - start_z
        new_start_z = current_max_height + 1
        new_end_z = new_start_z + brick_height
        
        settled_bricks.append((start_x, start_y, new_start_z, end_x, end_y, new_end_z))
        
        # Determine relationships
        for x in range(start_x, end_x + 1):
            for y in range(start_y, end_y + 1):
                height, top_brick_index = height_map.get((x, y), (0, -1))
                
                if height != current_max_height or top_brick_index == 1:
                    continue

                if brick_index not in supported_by:
                    supported_by[brick_index] = set()

                supported_by[brick_index].add(top_brick_index)
                
                if top_brick_index not in supports:
                    supports[top_brick_index] = set()

                supports[top_brick_index].add(brick_index)
        
        # Update height map
        for x in range(start_x, end_x + 1):
            for y in range(start_y, end_y + 1):
                height_map[(x, y)] = (new_end_z, brick_index)

    # Part 1: Count safe removals
    def is_safe_to_remove(brick_index):
        supported_bricks = supports.get(brick_index, set())

        for supported_brick_index in supported_bricks:
            supporters = supported_by.get(supported_brick_index, set())

            if len(supporters) == 1:
                return False
            
        return True

    safe_removal_count = sum(1 for i in range(len(settled_bricks)) if is_safe_to_remove(i))

    # Part 2: Count chain reactions
    def count_falling_bricks(disintegrated_brick_index):
        falling_bricks = set([disintegrated_brick_index])
        
        for other_brick_index in range(disintegrated_brick_index + 1, len(settled_bricks)):
            supporters = supported_by.get(other_brick_index, set())

            if supporters and supporters.issubset(falling_bricks):
                falling_bricks.add(other_brick_index)
                
        return len(falling_bricks) - 1

    total_chain_reaction_count = sum(count_falling_bricks(i) for i in range(len(settled_bricks)))

    return safe_removal_count, total_chain_reaction_count

In [3]:
part1, part2 = solve_puzzles()
print(f"Part 1: {part1}")
print(f"Part 2: {part2}")

Part 1: 527
Part 2: 100376
